### 1. Import libraries

In [1]:
import pandas as pd
import json
from datetime import datetime
import os

### 2. Load the raw data from extract stage

In [2]:
# Load the raw data from extract stage
with open('../data/raw_extract.json', 'r') as f:
    raw_data = json.load(f)

print(f"Loaded raw data for {len(raw_data)} cities")
print(f"Cities: {list(raw_data.keys())}")

Loaded raw data for 6 cities
Cities: ['London', 'New York', 'Tokyo', 'Sydney', 'Cape Town', 'Mumbai']


### 3. Inspect raw data structure

In [3]:
# Pick a city to inspect the structure
sample_city = list(raw_data.keys())[0]
print(f"Inspecting {sample_city} data structure")
print("=" * 50)
print(f"Main fields: {list(raw_data[sample_city].keys())}")
print("\nMain weather data:")
print(f"  - temp: {raw_data[sample_city]['main']['temp']}")
print(f"  - feels_like: {raw_data[sample_city]['main']['feels_like']}")
print(f"  - humidity: {raw_data[sample_city]['main']['humidity']}")
print(f"  - pressure: {raw_data[sample_city]['main']['pressure']}")
print("\nWeather description:")
print(f"  - main: {raw_data[sample_city]['weather'][0]['main']}")
print(f"  - description: {raw_data[sample_city]['weather'][0]['description']}")
print("\nWind data:")
print(f"  - speed: {raw_data[sample_city]['wind']['speed']}")
print("\nClouds:")
print(f"  - all: {raw_data[sample_city]['clouds']['all']}")
print("\nSystem info:")
print(f"  - country: {raw_data[sample_city]['sys']['country']}")

Inspecting London data structure
Main fields: ['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod']

Main weather data:
  - temp: 15.87
  - feels_like: 14.45
  - humidity: 36
  - pressure: 1024

Weather description:
  - main: Clouds
  - description: overcast clouds

Wind data:
  - speed: 2.68

Clouds:
  - all: 88

System info:
  - country: GB


### 4. Create transformation function

In [6]:
def transform_weather(raw_json, city_name):
    """
    Transform raw API data to clean structured format
    
    Args:
        raw_json: Raw API response
        city_name: City name
    
    Returns:
        dict: Cleaned data or None if raw_json is None
    """
    if not raw_json:
        return None
    
    transformed = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'city': city_name,
        'country': raw_json['sys']['country'],
        'temperature_c': raw_json['main']['temp'],
        'feels_like_c': raw_json['main']['feels_like'],
        'temp_min_c': raw_json['main']['temp_min'],
        'temp_max_c': raw_json['main']['temp_max'],
        'humidity_pct': raw_json['main']['humidity'],
        'pressure_hpa': raw_json['main']['pressure'],
        'weather_main': raw_json['weather'][0]['main'],
        'weather_description': raw_json['weather'][0]['description'],
        'wind_speed_ms': raw_json['wind']['speed'],
        'wind_deg': raw_json['wind'].get('deg', 0),
        'clouds_pct': raw_json['clouds']['all'],
        'visibility_m': raw_json.get('visibility', 0)
    }
    
    return transformed

### 5. Transform all cities

In [7]:
# Transform all cities
clean_data = []
failed_cities = []

print("Transforming data...")
print("-" * 40)

for city, data in raw_data.items():
    if data:
        transformed = transform_weather(data, city)
        clean_data.append(transformed)
        print(f"SUCCESS: {city:15} | {transformed['temperature_c']:5.1f}C | {transformed['weather_description']}")
    else:
        failed_cities.append(city)
        print(f"FAILED:  {city:15} | No data to transform")

print("-" * 40)
print(f"Transformation complete!")
print(f"   Successful: {len(clean_data)} cities")
print(f"   Failed: {len(failed_cities)} cities")

Transforming data...
----------------------------------------
SUCCESS: London          |  15.9C | overcast clouds
SUCCESS: New York        |  11.7C | mist
SUCCESS: Tokyo           |  17.5C | broken clouds
SUCCESS: Sydney          |  12.4C | broken clouds
SUCCESS: Cape Town       |  14.2C | overcast clouds
SUCCESS: Mumbai          |  30.0C | haze
----------------------------------------
Transformation complete!
   Successful: 6 cities
   Failed: 0 cities


### 6. Convert to DataFrame

In [8]:
# Create pandas DataFrame
df = pd.DataFrame(clean_data)

print("Transformed data as DataFrame:")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")
print("\nFirst 3 rows:")
df.head(3)

Transformed data as DataFrame:
   Shape: (6, 15)
   Columns: ['timestamp', 'city', 'country', 'temperature_c', 'feels_like_c', 'temp_min_c', 'temp_max_c', 'humidity_pct', 'pressure_hpa', 'weather_main', 'weather_description', 'wind_speed_ms', 'wind_deg', 'clouds_pct', 'visibility_m']

First 3 rows:


,timestamp,city,country,temperature_c,feels_like_c,temp_min_c,temp_max_c,humidity_pct,pressure_hpa,weather_main,weather_description,wind_speed_ms,wind_deg,clouds_pct,visibility_m
0,2026-04-19 17:18:24,London,GB,15.87,14.45,14.44,16.83,36,1024,Clouds,overcast clouds,2.68,50,88,10000
1,2026-04-19 17:18:24,New York,US,11.69,10.79,10.32,12.34,72,1011,Mist,mist,10.29,330,100,10000
2,2026-04-19 17:18:24,Tokyo,JP,17.46,17.27,15.92,18.24,77,1017,Clouds,broken clouds,3.60,70,75,10000


### 7. Check data types

In [9]:
# Check data types of each column
print("Data types:")
print(df.dtypes)
print("\nData types summary:")
print(df.info())

Data types:
timestamp                  str
city                       str
country                    str
temperature_c          float64
feels_like_c           float64
temp_min_c             float64
temp_max_c             float64
humidity_pct             int64
pressure_hpa             int64
weather_main               str
weather_description        str
wind_speed_ms          float64
wind_deg                 int64
clouds_pct               int64
visibility_m             int64
dtype: object

Data types summary:
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   timestamp            6 non-null      str    
 1   city                 6 non-null      str    
 2   country              6 non-null      str    
 3   temperature_c        6 non-null      float64
 4   feels_like_c         6 non-null      float64
 5   temp_min_c           6 non-null      float6

### 8. Data quality checks

In [10]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

print("\nBasic statistics for numeric columns:")
df[['temperature_c', 'feels_like_c', 'humidity_pct', 'wind_speed_ms', 'clouds_pct']].describe()

Missing values per column:
timestamp              0
city                   0
country                0
temperature_c          0
feels_like_c           0
temp_min_c             0
temp_max_c             0
humidity_pct           0
pressure_hpa           0
weather_main           0
weather_description    0
wind_speed_ms          0
wind_deg               0
clouds_pct             0
visibility_m           0
dtype: int64

Basic statistics for numeric columns:


,temperature_c,feels_like_c,humidity_pct,wind_speed_ms,clouds_pct
count,6.000000,6.000000,6.000000,6.000000,6.000000
mean,16.938333,16.991667,67.166667,4.991667,80.333333
std,6.741191,8.670872,15.854547,2.995379,22.312926
min,11.690000,10.790000,36.000000,2.570000,40.000000
25%,12.865000,12.215000,67.500000,2.910000,76.000000
50%,15.035000,14.030000,73.000000,3.860000,83.500000
75%,17.062500,16.565000,76.250000,6.047500,97.000000
max,29.990000,34.080000,78.000000,10.290000,100.000000


### 9. Add derived columns

In [11]:
# Add calculated/derived columns
df['temp_fahrenheit'] = round(df['temperature_c'] * 9/5 + 32, 1)
df['temp_range_c'] = round(df['temp_max_c'] - df['temp_min_c'], 1)
df['heat_index_c'] = round(
    df['temperature_c'] - (0.55 - 0.0055 * df['humidity_pct']) * (df['temperature_c'] - 14.5), 1
)

print("Added derived columns:")
print(f"New columns: {list(df.columns)}")
print("\nSample with new columns:")
df[['city', 'temperature_c', 'temp_fahrenheit', 'humidity_pct', 'heat_index_c']].head()

Added derived columns:
New columns: ['timestamp', 'city', 'country', 'temperature_c', 'feels_like_c', 'temp_min_c', 'temp_max_c', 'humidity_pct', 'pressure_hpa', 'weather_main', 'weather_description', 'wind_speed_ms', 'wind_deg', 'clouds_pct', 'visibility_m', 'temp_fahrenheit', 'temp_range_c', 'heat_index_c']

Sample with new columns:


,city,temperature_c,temp_fahrenheit,humidity_pct,heat_index_c
0,London,15.87,60.6,36,15.4
1,New York,11.69,53.0,72,12.1
2,Tokyo,17.46,63.4,77,17.1
3,Sydney,12.42,54.4,78,12.7
4,Cape Town,14.20,57.6,74,14.2


### 10. Sort and organize data

In [12]:
# Sort by temperature (hottest to coldest)
df_sorted = df.sort_values('temperature_c', ascending=False)

print("Cities sorted by temperature (hottest to coldest):")
print("-" * 50)
for idx, row in df_sorted.iterrows():
    print(f"{row['city']:15} | {row['temperature_c']:5.1f}C | {row['weather_description']}")

Cities sorted by temperature (hottest to coldest):
--------------------------------------------------
Mumbai          |  30.0C | haze
Tokyo           |  17.5C | broken clouds
London          |  15.9C | overcast clouds
Cape Town       |  14.2C | overcast clouds
Sydney          |  12.4C | broken clouds
New York        |  11.7C | mist


### 11. Create data directory and save transformed data

In [14]:
# Create data directory if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Save as CSV
df.to_csv('../data/transformed_weather.csv', index=False)
print("CSV saved to: data/transformed_weather.csv")

# Save as JSON
df.to_json('../data/transformed_weather.json', orient='records', indent=2)
print("JSON saved to: data/transformed_weather.json")

# Save as Excel (optional, requires openpyxl)
try:
    df.to_excel('../data/transformed_weather.xlsx', index=False)
    print("Excel saved to: data/transformed_weather.xlsx")
except ImportError:
    print("Excel export skipped (openpyxl not installed)")

print(f"\nRecords saved: {len(df)}")

CSV saved to: data/transformed_weather.csv
JSON saved to: data/transformed_weather.json
Excel export skipped (openpyxl not installed)

Records saved: 6


### 12. Final summary statistics

In [15]:
print("Transformation Stage Complete!")
print("=" * 50)
print(f"Total cities processed: {len(raw_data)}")
print(f"Records transformed: {len(df)}")
print(f"Columns created: {len(df.columns)}")
print("\nOutput files created:")
print("  - data/transformed_weather.csv")
print("  - data/transformed_weather.json")
print("\nTemperature Summary:")
print(f"  Hottest city: {df.loc[df['temperature_c'].idxmax(), 'city']} - {df['temperature_c'].max():.1f}C")
print(f"  Coldest city: {df.loc[df['temperature_c'].idxmin(), 'city']} - {df['temperature_c'].min():.1f}C")
print(f"  Average temperature: {df['temperature_c'].mean():.1f}C")
print("\nHumidity Summary:")
print(f"  Most humid: {df.loc[df['humidity_pct'].idxmax(), 'city']} - {df['humidity_pct'].max()}%")
print(f"  Least humid: {df.loc[df['humidity_pct'].idxmin(), 'city']} - {df['humidity_pct'].min()}%")
print("\nReady for next stage: 3_load.ipynb")

Transformation Stage Complete!
Total cities processed: 6
Records transformed: 6
Columns created: 18

Output files created:
  - data/transformed_weather.csv
  - data/transformed_weather.json

Temperature Summary:
  Hottest city: Mumbai - 30.0C
  Coldest city: New York - 11.7C
  Average temperature: 16.9C

Humidity Summary:
  Most humid: Sydney - 78%
  Least humid: London - 36%

Ready for next stage: 3_load.ipynb
